In [1]:
import os
import cv2

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from config import WLASL_RAW_DATA, HOW2SIGN_RAW_DATA
from src.utils import HandDetection, PoseDetection, FaceDetection

video_id = "_0fO5ETSwyg_0-5-rgb_front"
input_video = os.path.join(HOW2SIGN_RAW_DATA, f"{video_id}.mp4")
output_video = os.path.join(r"D:\SignDetection", f"{video_id}_skeleton.mp4")


cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")


hand_detection = HandDetection(
    min_hand_detection_confidence=0.2
)

pose_detection = PoseDetection(
    min_pose_detection_confidence=0.2
)

face_detection = FaceDetection()


fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 25


width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"width: {width}, height: {height}")
print(f"fps: {fps}")
print(f"Output: {output_video}")


# =========================
# Video Writer
# =========================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError("Cannot open video writer.")


frame_index = 0

while True:

    success, frame = cap.read()

    if not success:
        print("End of video.")
        break

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    timestamp_ms = int(
        frame_index * 1000 / fps
    )


    detection_hand_results = hand_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    detection_pose_results = pose_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    detection_face_results = face_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )

    output = hand_detection.draw_landmarks_on_image(
        rgb_frame.copy(),
        detection_hand_results
    )
    print(detection_hand_results)

    output = pose_detection.draw_landmarks_on_image(
        output,
        detection_pose_results
    )

    output = face_detection.draw_lips_on_image(
        output,
        detection_face_results
    )

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )


    writer.write(output)

    cv2.imshow(
        "Landmarks + Lips",
        output
    )


    frame_index += 1


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# =========================
# Release
# =========================

cap.release()
writer.release()

cv2.destroyAllWindows()

hand_detection.close()

print("Done.")
print(f"Saved video to: {output_video}")

width: 1280, height: 720
fps: 24.0
Output: D:\SignDetection\_0fO5ETSwyg_0-5-rgb_front_skeleton.mp4
HandLandmarkerResult(handedness=[[Category(index=0, score=0.9949131011962891, display_name='Right', category_name='Right')], [Category(index=1, score=0.9980476498603821, display_name='Left', category_name='Left')]], hand_landmarks=[[NormalizedLandmark(x=0.4735509753227234, y=0.5146428942680359, z=3.4498299328333815e-08, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.48732295632362366, y=0.48744937777519226, z=0.001444241264835, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.5059669017791748, y=0.47830283641815186, z=0.00037629727739840746, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.5208157896995544, y=0.4832925796508789, z=-0.0020970951300114393, visibility=None, presence=None, name=None), NormalizedLandmark(x=0.5294287204742432, y=0.49564963579177856, z=-0.00396726792678237, visibility=None, presence=None, name=None), Normal